# Flow Chart Pengerjaan:
1. Cover Image
2. DCT & Quantization
3. Block Smoothness Estimation & Sorting
4. Zigzag Scan
5. NACP Construction
6. Adaptive Hexagonal Payload Assignment
7. Hexagonal Turtle Shell Embedding
8. Stego DCT Coefficients
9. Entropy Coding
10. Stego Image

In [2]:
!pip install jpeglib numpy matplotlib opencv-python-headless scipy scikit-image seaborn pandas tqdm import-ipynb 


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
from PIL import Image
from performance import psnr, fsi, ssim
from zigzag import zigzag, inverse_zigzag
from math import ceil, floor, log2, log10, sqrt
import copy
import cv2
import jpeglib
import numpy as np
import pandas as pd
import import_ipynb
import matplotlib.pyplot as plt
import TurtleShell
import FrequencyDomain as FD

In [4]:
# Global variable
scale_factor = 1.0
threshold = 150
global nacp_before_embed
nacp_before_embed = []
global nacp_after_embed
nacp_after_embed = []
global nacp_before_extract 
nacp_before_extract = []

In [5]:
# First initialize the turtle shells for both 8N and 17N modes
_, shells, cell_to_shells = TurtleShell.init(mode="8N")
_, shells_17, cell_to_shells_17 = TurtleShell.init(mode="17N")

In [6]:
def change_image_QF_high_quality(image_path, target_qf):
    im = jpeglib.read_dct(image_path)
    old_qt = im.qt[0]

    # From QF 100 to target QF
    dequantized = im.Y.astype(np.float64) * old_qt
    new_coefficients = np.round(dequantized / FD.custom_q_mat(target_qf)).astype(np.int16)
    
    # Update image object
    im.Y[:] = new_coefficients
    im.qt[0] = FD.custom_q_mat(target_qf)

    dequantized = im.Y.astype(np.float64) * FD.custom_q_mat(target_qf)
    im.Y[:] = np.round(dequantized / FD.custom_q_mat(100)).astype(np.int16)
    im.qt[0] = FD.custom_q_mat(100)
    
    ori_path =  image_path.split(".jpeg")[0]
    output_path = f"{ori_path}_qf{target_qf}_hd.jpeg"
    im.write_dct(output_path)

In [7]:
def change_image_QF(image_path, target_qf):
    im = jpeglib.read_dct(image_path)
    old_qt = im.qt[0]

    # From QF 100 to target QF
    dequantized = im.Y.astype(np.float64) * old_qt
    new_coefficients = np.round(dequantized / FD.custom_q_mat(target_qf)).astype(np.int16)
    
    # Update image object
    im.Y[:] = new_coefficients
    im.qt[0] = FD.custom_q_mat(target_qf)
    
    ori_path =  image_path.split(".jpeg")[0]
    output_path = f"{ori_path}_qf{target_qf}.jpeg"
    im.write_dct(output_path)

In [8]:
def get_quantized_coefficients(image_path):
    im = jpeglib.read_dct(image_path)
    num_v_blocks, num_h_blocks, _, _  = im.Y.shape
    sorted_coeffs = []
    for i in range(num_v_blocks):
        for j in range(num_h_blocks):
            block = im.Y[i, j]
            zigzag_coeffs = zigzag(block)
            sorted_coeffs.append(zigzag_coeffs)
    return sorted_coeffs

In [9]:
def get_compress_coeff(image_path):
    get_qf = image_path.find("_qf")
    target_qf = int(image_path[get_qf+3:get_qf+5])
    im = jpeglib.read_dct(image_path)
    print(f"Target QF: {target_qf}")
    print(f"Scale Factor: {scale_factor}")
    print(f"Image QT: \n{im.qt[0]}")
    num_v_blocks, num_h_blocks, _, _  = im.Y.shape
    sorted_coeffs = []
    for i in range(num_v_blocks):
        for j in range(num_h_blocks):
            block = im.Y[i, j] * im.qt[0] * scale_factor / FD.custom_q_mat(target_qf)
            zigzag_coeffs = zigzag(block)
            sorted_coeffs.append(zigzag_coeffs)
    return sorted_coeffs

In [10]:
def convert_tiff_to_jpeg(tiff_path, jpeg_path, quality=100):
    with Image.open(tiff_path) as img:
        # rgb_img = img.convert('RGB')  # Convert to RGB if it's not already
        # rgb_img.save(jpeg_path, 'JPEG', quality=quality, subsampling=0, optimize=False)
        gray_img = img.convert('L')  # Convert to grayscale
        gray_img.save(jpeg_path, 'JPEG', quality=quality, subsampling=0, optimize=False)
    print(f"Converted {tiff_path} to {jpeg_path} with quality {quality}")

In [11]:
def sort_smoothness(smoothness_list):
    smoothness_list.sort(key=lambda x: (-x[1], x[2]))
    return smoothness_list

In [12]:
def block_smoothness_estimation(image):
    im = jpeglib.read_dct(image)
    h, w, _, _ = im.Y.shape
    smoothness_block = []
    total_ec = 0
    total_zero_count = 0
    for i in range(h):
        for j in range(w):
            block = im.Y[i, j]
            print( block)
            block_1d = block.flatten()
            block_1d = block_1d[1:]  # AC coefficients 
            zero_count = np.sum(block_1d == 0)
            non_zero_sum = np.sum(abs(block_1d[block_1d != 0]))
            non_zero_indices = np.nonzero(block_1d)[0]
            capable_bits = 4 if zero_count == 0 else 3
            total_ec += len(non_zero_indices) // 2 * capable_bits
            total_zero_count += zero_count
            smoothness_block.append(((i, j), zero_count, non_zero_sum))

    print(f"Total blocks: {h * w}")
    print(f"Total embedding capacity (estimated): {total_ec} bits")
    print(f"Total zero count: {total_zero_count}")
    return total_ec

def block_smoothness(image):
    im = jpeglib.read_dct(image)
    h, w, _, _ = im.Y.shape
    q_table = im.qt[0]
    smoothness_block = []
    smoothness_score = []
    sum_nacp = 0
    for i in range(h):
        for j in range(w):
            block = im.Y[i, j]
            block_1d = block.flatten()
            block_1d = block_1d[1:]  # AC coefficients
            ac_block = block.copy()
            ac_block[0, 0] = 0
            z_k = np.sum(ac_block == 0)
            E_k = np.sum((ac_block != 0) * (q_table ** 2))
            S_k = z_k + float(z_k / E_k)
            non_zero_indices = np.nonzero(block_1d)[0]
            sum_nacp += non_zero_indices.size
            smoothness_block.append(((i, j), z_k, E_k, S_k))
            smoothness_score.append(((i, j), S_k))
    
    mean_score = np.mean([score for _, score in smoothness_score])
    total_ec = sum_nacp // 2 * 3
    print(f"Total blocks: {h * w}")
    print(f"Total NACP: {sum_nacp}")
    print(f"Max embedding capacity (estimated): {total_ec} bits")
    # print(f"Average smoothness score: {mean_score}")
    return smoothness_block, smoothness_score, mean_score, total_ec

In [13]:
def invariant_ac_smoothness(image_path, threshold_eob):
    coeffs = get_quantized_coefficients(image_path)
    smoothness_block = []
    for idx in range(len(coeffs)):
        sum_z_k = 0
        sum_ac_k = 0
        for k in range(threshold_eob + 1, 64):
            if coeffs[idx][k] == 0:
                sum_z_k += 1
            sum_ac_k += abs(coeffs[idx][k])
        smoothness_block.append(((idx), sum_z_k, sum_ac_k))
    s_smoothness_block = sort_smoothness(smoothness_block)
    return smoothness_block, s_smoothness_block

In [14]:
def get_nacp(sorted_coefficients):
    valid_nacp = []
    for zigzag_coeff in sorted_coefficients:
        ac_coeffs = zigzag_coeff[1:] # AC coefficients
        non_zero_indices = np.nonzero(np.abs(ac_coeffs) >= 1)[0]
        non_zero_ac = [ac_coeffs[i] for i in non_zero_indices]

        for i in range(0, len(non_zero_ac) - 1, 2):
            x = int(non_zero_ac[i])
            y = int(non_zero_ac[i+1])
            if x != 0 and y != 0:
                valid_nacp.append((x, y))

    return valid_nacp

def get_nacp_2(sorted_coefficients):
    valid_nacp = []
    for zigzag_coeff in sorted_coefficients:
        ac_coeffs = zigzag_coeff[1:] # AC coefficients
        non_zero_indices = np.nonzero(np.abs(ac_coeffs) > 1)[0]
        non_zero_ac = [ac_coeffs[i] for i in non_zero_indices]

        for i in range(0, len(non_zero_ac) - 1, 2):
            x = int(non_zero_ac[i])
            y = int(non_zero_ac[i+1])
            if x != 0 and y != 0:
                valid_nacp.append((x, y))

    return valid_nacp

In [15]:
def replace_nacp(sorted_coefficients, nacp_coords):
    pair_index = 0

    for zigzag_coeff in sorted_coefficients:
        ac_coeffs = zigzag_coeff[1:]  
        non_zero_indices = np.nonzero(np.abs(ac_coeffs) >= 1)[0]
        non_zero_ac = [ac_coeffs[i] for i in non_zero_indices]

        for idx in range(0, len(non_zero_ac) - 1, 2):
            if pair_index < len(nacp_coords):
                new_x, new_y = nacp_coords[pair_index]
                i1, i2 = non_zero_indices[idx], non_zero_indices[idx + 1]
                ac_coeffs[i1] = float(new_x)
                ac_coeffs[i2] = float(new_y)
                pair_index += 1
            else: break
        zigzag_coeff[1:] = ac_coeffs

    return sorted_coefficients

def replace_nacp_2(sorted_coefficients, nacp_coords):
    pair_index = 0
    for zigzag_coeff in sorted_coefficients:
        ac_part = zigzag_coeff[1:]
        rel_non_zero_indices = np.nonzero(np.abs(ac_part) > 1)[0]        
        for idx in range(0, len(rel_non_zero_indices) - 1, 2):
            if pair_index < len(nacp_coords):
                new_x, new_y = nacp_coords[pair_index]                
                zigzag_coeff[rel_non_zero_indices[idx] + 1] = float(new_x)
                zigzag_coeff[rel_non_zero_indices[idx+1] + 1] = float(new_y)
                pair_index += 1
            else: break
    return sorted_coefficients

In [16]:
def construct_stego_file(image_path, new_coeffs):
    im = jpeglib.read_dct(image_path)
    num_v_blocks, num_h_blocks, v_block_size, h_block_size  = im.Y.shape
    idx = 0
    
    for i in range(num_v_blocks):
        for j in range(num_h_blocks):
            block_coeffs = new_coeffs[idx]
            block = inverse_zigzag(block_coeffs, v_block_size, h_block_size)
            im.Y[i, j] = block
            idx += 1

    output_path = "stego-images/stego_" + image_path.split("/")[-1]
    print(f"Image with secret data is saved to {output_path}")
    im.write_dct(output_path)

def construct_compress_stego_file(image_path, new_coeffs):
    get_qf = image_path.find("_qf")
    target_qf = int(image_path[get_qf+3:get_qf+5])
    im = jpeglib.read_dct(image_path)
    num_v_blocks, num_h_blocks, v_block_size, h_block_size  = im.Y.shape
    idx = 0
    
    for i in range(num_v_blocks):
        for j in range(num_h_blocks):
            block_coeffs = new_coeffs[idx]
            block = inverse_zigzag(block_coeffs, v_block_size, h_block_size)
            block = (block.astype(np.float64) * FD.custom_q_mat(target_qf)) / (im.qt[0].astype(np.float64) * scale_factor)
            im.Y[i, j] = block
            idx += 1

    output_path = "stego-images/stego_" + image_path.split("/")[-1]
    print(f"Image with secret data is saved to {output_path}")
    im.write_dct(output_path)

In [17]:
def data_hiding_process(secret_data, nacp_coord="", mode="8N"):
    if secret_data == "": return nacp_coord
    bit = 3 if mode == "8N" else 4
    secret_data = secret_data + '\0'
    data_bin = ''.join(format(ord(c), '08b') for c in secret_data)
    lendata = len(data_bin)
    print(f"Secret Data: {secret_data}")
    print(f"Panjang Bit Secret Data: {lendata}")

    decimals = []
    for i in range(0, len(data_bin), bit):
        group = data_bin[i:i+bit].ljust(bit, '0')
        decimals.append(int(group, 2))

    _, shells, cell_to_shells = TurtleShell.init(mode) 
    if len(decimals) > len(nacp_coord):
        print("Warning: Not enough NACP coordinates to embed all data.")
        
    for i in range(len(decimals)):
        x, y = nacp_coord[i]
        if TurtleShell.get_hex_matrix_value(x, y, mode) == decimals[i]:
            nacp_coord[i] = (x, y)
        else:
            # shell_coords = TurtleShell.get_kxk_nearest_signed(x, y, bit)
            _, shell_coords = TurtleShell.get_shell_coords(x, y, shells, cell_to_shells)
            found = TurtleShell.find_corresponding_val(shell_coords, decimals[i], nacp_coord[i], mode)
            nacp_coord[i] = found
            
    return nacp_coord

In [18]:
def data_extract_process(nacp_coord, mode="8N"):
    extracted_data = ""
    data_bits = ""

    for i, (x, y) in enumerate(nacp_coord):
        val = TurtleShell.get_hex_matrix_value(x, y, mode=mode)
        bit = 3 if mode == "8N" else 4
        bits = format(val & ((1 << bit) - 1), f'0{bit}b')
        data_bits += bits

        while len(data_bits) >= 8:
            byte = data_bits[:8]
            char_val = int(byte, 2)
            if char_val == 0: # Null terminator ASCII
                return extracted_data
            try:
                char = chr(char_val)
                extracted_data += char
            except:
                return extracted_data
            data_bits = data_bits[8:]

    return extracted_data

In [19]:
def encode(image_path, message_bits):
    sorted_coeffs = get_quantized_coefficients(image_path)
    nacp_coords = get_nacp(sorted_coeffs)
    print(nacp_coords)
    print(f"NACP Length: {len(nacp_coords)}")
    print(f"Total EC: {len(nacp_coords * 3)}")
    modified_nacp_coords = data_hiding_process(message_bits, nacp_coords, mode="8N")
    modified_coeffs = replace_nacp(sorted_coeffs, modified_nacp_coords)
    print(modified_nacp_coords)
    construct_stego_file(image_path, modified_coeffs)
    print("Data embedding completed.")

def encode_2(image_path, data, qf):
    image = Image.open(image_path).convert('L')
    stegoimg = image.copy()
    img_arr = np.array(stegoimg)
    q_mat = FD.custom_q_mat(qf)
    sorted_coefficients = FD.transform_to_freq(img_arr, q_mat)
    nacp_coords = get_nacp(sorted_coefficients)
    modified_nacp_coords = data_hiding_process(data, nacp_coords)
    modified_coeffs = replace_nacp(sorted_coefficients, modified_nacp_coords)
    np.save("modified_coefficients.npy", modified_coeffs)

def encode_4(image_path, secret_data):
    sorted_coeffs = get_compress_coeff(image_path)
    nacp_coords = get_nacp(sorted_coeffs)
    global nacp_before_embed
    nacp_before_embed = nacp_coords.copy()
    print(f"NACP Length: {len(nacp_coords)}")
    print(f"Total EC: {len(nacp_coords * 3)}")
    modified_nacp_coords = data_hiding_process(secret_data, nacp_coords, mode="8N")
    modified_coeffs = replace_nacp(sorted_coeffs, modified_nacp_coords)
    global nacp_after_embed
    nacp_after_embed = modified_nacp_coords.copy()
    construct_compress_stego_file(image_path, modified_coeffs)
    
# Proposed Method - Adaptive Payload in Multi Turtle Shell Embedding
def encode_3(image_path, secret_data):
    _, smoothness_score, mean_score, total_ec = block_smoothness(image_path)
    smoothness_list = sorted(smoothness_score, key=lambda x: -x[1])
    secret_data += '\0'
    data_bin = ''.join(format(ord(c), '08b') for c in secret_data)
    lendata = len(data_bin)

    im = jpeglib.read_dct(image_path)
    for (block_i, block_j), score in smoothness_list:
        # print(f"Processing block ({block_i}, {block_j}) with smoothness score {score}")
        if lendata <= 0: break
        # N = 8 if score >= threshold else 17
        N = 8
        t = 3 if N == 8 else 4
        mode = f"{N}N"
        block = im.Y[block_i, block_j]
        zigzag_coeffs = zigzag(block)
        ac_coeffs = zigzag_coeffs[1:]
        non_zero_indices = np.nonzero(np.abs(ac_coeffs) >= 1)[0]
        choosen_shells = shells if N == 8 else shells_17
        choosen_cell_to_shells = cell_to_shells if N == 8 else cell_to_shells_17
        for idx in range(0, len(non_zero_indices) - 1, 2):
            if lendata <= 0: break
            x = int(ac_coeffs[non_zero_indices[idx]])
            y = int(ac_coeffs[non_zero_indices[idx + 1]])
            bits = data_bin[:t].ljust(t, '0') 
            data_bin = data_bin[t:]
            lendata -= t
            target_val = int(bits, 2)
            val_int = TurtleShell.get_hex_matrix_value(x, y, mode=mode)
            if target_val != val_int:
                shell_coords = TurtleShell.get_kxk_nearest_signed(x, y, t)
                # _, shell_coords = TurtleShell.get_shell_coords(x, y, choosen_shells, choosen_cell_to_shells)
                x, y = TurtleShell.find_corresponding_val(shell_coords, target_val, (x, y), mode=mode)
            ac_coeffs[non_zero_indices[idx]] = float(x)
            ac_coeffs[non_zero_indices[idx + 1]] = float(y)

        zigzag_coeffs[1:] = ac_coeffs
        zigzag_coeffs[0] = block[0, 0]  
        im.Y[block_i, block_j] = inverse_zigzag(zigzag_coeffs, 8, 8)

    output_path = "stego-images/stego_" + image_path.split("/")[-1]
    print(f"Data embedding completed. Stego image saved to {output_path}")
    im.write_dct(output_path)
    return total_ec

In [20]:
def decode(stego_image_path):
    sorted_coeffs = get_quantized_coefficients(stego_image_path)
    nacp_coords = get_nacp(sorted_coeffs)
    print(nacp_coords)
    extracted_data = data_extract_process(nacp_coords, mode="8N")
    return extracted_data

def decode_2(stego_file):
    modified_coeffs = np.load(stego_file, allow_pickle=True)
    nacp_coords = get_nacp(modified_coeffs)
    extracted_data = data_extract_process(nacp_coords)
    return extracted_data

def decode_4(stego_image_path):
    sorted_coeffs = get_compress_coeff(stego_image_path)
    nacp_coords = get_nacp(sorted_coeffs)
    global nacp_before_extract
    nacp_before_extract = nacp_coords.copy()
    extracted_data = data_extract_process(nacp_coords, mode="8N")
    return extracted_data

# Proposed Method - Adaptive Payload in Multi Turtle Shell Embedding
def decode_3(stego_image_path):
    _, smoothness_score, mean_score, total_ec = block_smoothness(stego_image_path)
    smoothness_list = sorted(smoothness_score, key=lambda x: -x[1])
    im = jpeglib.read_dct(stego_image_path)
    bitstream = ""
    decoded_text = ""
    for (block_i, block_j), score in smoothness_list:
        # N = 8 if score >= threshold else 17
        N = 8
        t = 3 if N == 8 else 4
        mode = f"{N}N"
        block = im.Y[block_i, block_j]
        zigzag_coeffs = zigzag(block)
        ac_coeffs = zigzag_coeffs[1:]
        non_zero_indices = np.nonzero(np.abs(ac_coeffs) >= 1)[0]

        for idx in range(0, len(non_zero_indices) - 1, 2):
            x = int(ac_coeffs[non_zero_indices[idx]])
            y = int(ac_coeffs[non_zero_indices[idx + 1]])
            val = TurtleShell.get_hex_matrix_value(x, y, mode=mode)
            bits = format(val, f"0{t}b")
            bitstream += bits
            while len(bitstream) >= 8:
                byte = bitstream[:8]
                bitstream = bitstream[8:]
                char_val = int(byte, 2)
                if char_val == 0:   # Null terminator
                    return decoded_text
                decoded_text += chr(char_val)
    return decoded_text

In [21]:
def read_text_file(file_path):
    with open(file_path, 'r', encoding='utf-8') as file:
        content = file.read()
    return content

In [22]:
# cover_images = [
#     "cover-images/misc/boat.512.tiff",
#     "cover-images/misc/4.2.01.tiff",
#     "cover-images/misc/4.2.03.tiff",
#     "cover-images/misc/4.2.05.tiff",
#     "cover-images/misc/4.2.06.tiff",
#     "cover-images/misc/4.2.07.tiff"
# ]

# jpeg_images_path = [
#     "cover-images/boat.jpeg",
#     "cover-images/splash.jpeg",
#     "cover-images/baboon.jpeg",
#     "cover-images/airplane.jpeg",
#     "cover-images/lake.jpeg",
#     "cover-images/peppers.jpeg"
# ]

# # convert_tiff_to_jpeg("cover-images/misc/boat.512.tiff", "cover-images/boat_qf100.jpeg", quality=100)

# for tiff_path, jpeg_path in zip(cover_images, jpeg_images_path):
#     convert_tiff_to_jpeg(tiff_path, jpeg_path, quality=100)

In [23]:
pay_size = 1 * 1000
cover_folder = "cover-images/"
stego_folder = "stego-images/"
payload_folder = "payload/"
cover_image_path = f"baboon_qf50.jpeg"
stego_image_path = f"stego_{cover_image_path}"
data = read_text_file(f"{payload_folder}{pay_size}bits.txt")
encode_3(f"{cover_folder}{cover_image_path}", data)
# encode(f"{cover_folder}{cover_image_path}", data)

# secret_data = decode(f"{stego_folder}{stego_image_path}")
secret_data = decode_3(f"{stego_folder}{stego_image_path}")
# secret_data = decode_4(f"{stego_folder}{stego_image_path}")
print("Extracted Data:", secret_data) 

Total blocks: 4096
Total NACP: 63053
Max embedding capacity (estimated): 94578 bits
Data embedding completed. Stego image saved to stego-images/stego_baboon_qf50.jpeg
Total blocks: 4096
Total NACP: 63053
Max embedding capacity (estimated): 94578 bits
Extracted Data: Lorem ipsum dolor sit amet, consectetur adipiscing elit. Sed do eiusmod tempor incididunt ut labore et dolore magna aliqua. U


C:\Users\LENOVO\AppData\Local\Temp\ipykernel_30020\3882281584.py:42: RuntimeWarning: divide by zero encountered in divide
  S_k = z_k + float(z_k / E_k)


In [24]:
# Test performance metrics
psnr_value = psnr(f"{cover_folder}{cover_image_path}", f"{stego_folder}{stego_image_path}")
fsi_value = fsi(f"{cover_folder}{cover_image_path}", f"{stego_folder}{stego_image_path}")
ssim_value = ssim(f"{cover_folder}{cover_image_path}", f"{stego_folder}{stego_image_path}")
print(f"PSNR: {psnr_value} dB")
print(f"FSI: {fsi_value}")
print(f"SSIM: {ssim_value}")

Size cover: 45551
Size stego: 45659
PSNR: 50.46937367663537 dB
FSI: 108.0
SSIM: 0.9957648648173595


In [25]:
print(nacp_before_embed)
print(nacp_after_embed)
print(nacp_before_extract)
# diff between nacp_before_embed and nacp_after_embed
for i in range(min(len(nacp_before_embed), len(nacp_after_embed))):
    if nacp_before_embed[i] != nacp_after_embed[i]:
        print(f"Difference at index {i}: before embed {nacp_before_embed[i]}, after embed {nacp_after_embed[i]}")

# diff between nacp_after_embed and nacp_before_extract
for i in range(min(len(nacp_after_embed), len(nacp_before_extract))):
    if nacp_after_embed[i] != nacp_before_extract[i]:
        print(f"Difference at index {i}: after embed {nacp_after_embed[i]}, before extract {nacp_before_extract[i]}")

[]
[]
[]


In [26]:
def compare_spatial_frequency(cover_image_path, stego_image_path):
    cover = np.array(Image.open(cover_image_path).convert('L'), dtype=np.float64)
    stego = np.array(Image.open(stego_image_path).convert('L'), dtype=np.float64)

    spatial_difference = np.abs(cover - stego) ** 2

    cover_freq = get_quantized_coefficients(cover_image_path)
    stego_freq = get_quantized_coefficients(stego_image_path)
    freq_difference = np.array(cover_freq) - np.array(stego_freq)

    return freq_difference, spatial_difference

In [27]:
freq_diff, spatial_diff = compare_spatial_frequency(f"{cover_folder}{cover_image_path}", f"{stego_folder}{stego_image_path}")
print("Frequency Difference:")
for row in freq_diff:
    print(row)
print("Spatial Difference:")
for row in spatial_diff:
    print(row)
print("Max spatial diff:", spatial_diff.max())
print("Mean spatial diff:", spatial_diff.mean())
print("Max frequency diff:", np.max(freq_diff))
print("Mean frequency diff:", np.mean(freq_diff))

Frequency Difference:
[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.

In [28]:
# image_base = [
#     "cover-images/baboon.jpeg",
#     "cover-images/airplane.jpeg",
#     "cover-images/lake.jpeg",
#     "cover-images/peppers.jpeg",
#     "cover-images/splash.jpeg",
#     "cover-images/boat.jpeg"
# ]

# qf = [50, 60, 70, 80, 90]

# for img in image_base:
#     for q in qf:
#         change_image_QF(img, q)
#         change_image_QF_high_quality(img, q)

# Analisis Performance

In [29]:
# custom_quality = [50,60,70,80,90]
# payload_list = [1000, 2000, 3000, 4000, 5000, 6000, 7000, 8000, 9000, 10000, 11000, 12000, 13000, 14000, 15000]
# stego_image_folder = "stego-images/"
# cover_image_folder = "cover-images/"
# payload_folder = "payload/"
# cover_image_base = ["airplane", "baboon", "boat", "lake", "peppers", "splash"]

# df = pd.DataFrame(columns=['Image', 'Quality', 'PSNR', 'SSIM', 'FSI', 'Payload', 'Max EC', 'Success'])

# for img in cover_image_base:
#     for q in custom_quality:
#         for psize in payload_list:
#             detailed_img = f"{cover_image_folder}{img}_qf{q}_hd.jpeg"
#             print(f"Analyzing {detailed_img}")
#             data = read_text_file(f"{payload_folder}{psize}bits.txt")
#             total_ec = encode_3(f"{detailed_img}", data)
#             secret_data = decode_3(f"{stego_image_folder}stego_{img}_qf{q}_hd.jpeg")
#             psnr_value = psnr(detailed_img, f"{stego_image_folder}stego_{img}_qf{q}_hd.jpeg")
#             ssim_value = ssim(detailed_img, f"{stego_image_folder}stego_{img}_qf{q}_hd.jpeg")
#             fsi_value = fsi(detailed_img, f"{stego_image_folder}stego_{img}_qf{q}_hd.jpeg")
#             # print(f"After Encoding Secret Data : {secret_data}")
#             success = "Yes" if secret_data == data else "No"
#             df.loc[len(df)] = [img, q, psnr_value, ssim_value, fsi_value, psize, total_ec, success]

# df.to_csv("proposed_turtlesmooth_results_3.csv", index=False)
# df

# Analisis Image

In [ ]:
# custom_quality = [50,60,70,80,90]
# cover_image_base = ["airplane", "baboon", "boat", "lake", "peppers", "splash"]



In [31]:
custom_mse = 0.5625
psnr_val = 20 * np.log10(255.0 / np.sqrt(custom_mse))
print("PSNR calculated from custom MSE:", psnr_val, "dB")

PSNR calculated from custom MSE: 50.6295783408451 dB
